# Week 1 - Day 3: Build Your Own Agent Tools

> **Student notebook.** On Day 2 you watched, then typed. Today you write the **tools yourself**. Each tool cell is a **TODO** with step-by-step hints, followed by a **check** that prints `PASS` or `NOT YET`. Run the cells in order. A cell fails until the cells above it are done: that is normal.

Today we build a **trip-planner agent** with five tools. One is yesterday's, and **you write four**:

| Part | Tool | What it does | Source |
|---|---|---|---|
| **1** | `search_web` (given) | Finds pages: a title, a short snippet and a **link** | DuckDuckGo (no key) |
| **1** | `extract_page` (you write) | **Opens a link** and reads the part of the page you need | Tavily (free key) |
| **2** | `get_coordinates` (you write) | Turns a city name into a **latitude and longitude** | Open-Meteo (no key) |
| **2** | `get_weather` (you write) | Gives the **weather at** a latitude and longitude | Open-Meteo (no key) |
| **3** | `convert_currency` (you write) | Converts money, for example NPR to USD | open.er-api.com (no key) |
| **4** | the agent | Gives the five tools to one agent, with a system prompt | LangChain |
| **5** | limits and memory | A step limit, and a follow-up question that works | LangGraph |
| **6** | tests | The same three questions for everybody | |

On Day 2 the search tool returned only a short **snippet**. Today a **second tool** opens the page, so the agent can search first, then read.

## Setup
You need **two** free keys:
1. **Groq:** console.groq.com -> API Keys. This runs the model.
2. **Tavily:** tavily.com -> sign up, then copy the key from the dashboard. It is used only to **read pages**. The search (DuckDuckGo) needs no key.

Add them as secrets:
- **On Google Colab:** click the Secrets icon (key) in the left bar. Add `GROQ_API_KEY` and `TAVILY_API_KEY`, paste each key as the value, and switch **Notebook access** on for both.
- **On your own computer:** put both lines in a file called `.env` in this folder (or any folder above it):
  `GROQ_API_KEY=your_groq_key` and `TAVILY_API_KEY=your_tavily_key`

Your keys are like passwords. Never share them and never type them into the notebook.

In [ ]:
%pip install -q langchain langchain-groq ddgs tavily-python python-dotenv requests

In [ ]:
import os
import time
import requests
from datetime import date
from dotenv import find_dotenv, load_dotenv
from ddgs import DDGS
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient

load_dotenv(find_dotenv(usecwd=True), override=True)    # on your computer: reads your .env file
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "")

try:                                                     # on Google Colab: reads the keys from Secrets
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") or GROQ_API_KEY
    TAVILY_API_KEY = userdata.get("TAVILY_API_KEY") or TAVILY_API_KEY
except Exception:
    pass

if GROQ_API_KEY == "":
    raise RuntimeError("No GROQ_API_KEY found. Add it (Colab: Secrets, computer: .env file), then run this cell again.")
if TAVILY_API_KEY == "":
    raise RuntimeError("No TAVILY_API_KEY found. Add it (Colab: Secrets, computer: .env file), then run this cell again.")

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
THINKING = "low" if MODEL.startswith("openai/gpt-oss") else None    # "low" makes gpt-oss models answer faster

llm = ChatGroq(model=MODEL, api_key=GROQ_API_KEY, temperature=0.3, reasoning_effort=THINKING)
tavily = TavilyClient(api_key=TAVILY_API_KEY)      # the service our page-reading tool will use
today = str(date.today())                          # a model does not know today's date, so we will tell it
print("Ready. Model:", MODEL)

---
# Part 1 - Search, then read

A tool is a Python function with `@tool` on top. Remember the three things the model sees: the **name**, the **docstring** (the first line inside the function) and the **inputs**.

Yesterday's `search_web` finds pages, but it returns only a short **snippet** for each. To really **read** a page, the agent needs a second tool. So the agent works in two steps, and each step is its own tool:

```
question  ->  [ search_web ]  ->  a link  ->  [ extract_page ]  ->  the text of that page  ->  the answer
```
First, `search_web` (given). It is the tool you wrote on Day 2, so it is already here.

In [ ]:
@tool                                     # makes the function a tool the model can use
def search_web(query: str) -> str:
    """Search the web and return the top results with a short summary and a link for each."""   # the model reads this sentence
    # Step 1: ask DuckDuckGo for 3 results. If it fails, wait a moment and try once more.
    results = None
    for attempt in range(2):                      # the free search sometimes says "slow down", so we try twice
        try:
            results = DDGS().text(query, max_results=3)
            break                                 # it worked, so stop trying
        except Exception:
            time.sleep(2)                         # it failed, so wait 2 seconds

    # Step 2: if both tries failed, return a short message. A good tool never crashes.
    if results is None:
        return "Error: the search failed. Check the search words, or try again in a moment."

    # Step 3: the search worked but found nothing.
    if len(results) == 0:
        return "Error: nothing found for " + query + ". Try different words."

    # Step 4: one line per result: title, short summary (first 300 characters), link.
    lines = []
    for result in results:
        lines.append(result["title"] + ": " + result["body"][:300] + " (" + result["href"] + ")")
    return "\n".join(lines)

Run it by hand. Look at what comes back: a **short snippet and a link** for each result. That is all the agent knows after a search.

In [ ]:
print(search_web.invoke({"query": "things to do in Pokhara Nepal"}))

### Tool 1: `extract_page`

Now the tool **you** write. It takes a **link** and a **topic**, opens the page with Tavily's `extract`, and returns the text.

- A whole page can be huge: one test page had 95,000 characters, and the start of most pages is menus and images. So we pass the **topic** as `query=`: Tavily then returns only the parts of the page that are about the topic, as clean text (`format="text"`).
- We still keep at most the **first 2,500 characters**, to save tokens.
- A link that does not work gives an **empty** `"results"` list, so reading `pages[0]` raises an error. A badly written link (for example an empty one) makes Tavily raise an error. Both end in the **same `except`**, which returns an `Error:` text. You need only **one** `try / except` and no `if`.

In [ ]:
# TODO 1.1: write the tool.
#   - put @tool on the line above the function
#   - def extract_page(url: str, topic: str) -> str:
#   - the docstring (one sentence, FIRST line inside the function) tells the model what the tool does and where to get the link
#   - put EVERYTHING inside ONE try / except Exception, and in the except return "Error: that page could not be read. ..."
#   Inside the try:
#     - pages = tavily.extract(urls=[url], query=topic, format="text")["results"]
#       (urls must be a LIST, so write [url]; query=topic keeps the parts about the topic; format="text" gives clean text)
#     - return "Page: " + url + "\n" + pages[0]["raw_content"][:2500]      ([:2500] = the first 2500 characters)
#   You need no "if": a link that does not work gives an empty list, so pages[0] raises an error and the except catches it.

In [ ]:
# CHECK 1: run your tool by hand. No model is involved.
try:
    page = extract_page.invoke({"url": "https://en.wikipedia.org/wiki/Pokhara", "topic": "things to do"})
    print(page[:600])
    print("...")
    print("Characters returned:", len(page))

    if page.startswith("Error:"):
        print("NOT YET: the tool returned an error. Check that TAVILY_API_KEY is set, then run again.")
    elif len(page) < 500:
        print("NOT YET: the text is short. Did you pass query=topic to tavily.extract?")
    elif len(page) > 2700:
        print("NOT YET: the text is too long. Keep only the first 2500 characters: [:2500].")
    else:
        print("PASS: the tool opened the page and returned the text about the topic.")

    unreachable = extract_page.invoke({"url": "https://this-domain-does-not-exist-12345.com/x", "topic": "x"})
    print(unreachable)
    if unreachable.startswith("Error:"):
        print("PASS: a link that does not work returns an Error: text.")
    else:
        print("NOT YET: a link that does not work should return an Error: text. Check for an empty results list.")

    empty = extract_page.invoke({"url": "", "topic": "x"})
    if empty.startswith("Error:"):
        print("PASS: an empty link returns an Error: text, not a crash.")
    else:
        print("NOT YET: an empty link should return an Error: text. Put tavily.extract inside try / except.")
except NameError:
    print("NOT YET: write the tool in the cell above, and run that cell first.")

---
# Part 2 - Tools 2 and 3: coordinates, then weather

A weather service does not understand a city name. It wants **coordinates**: a latitude and a longitude. Finding the coordinates and getting the weather are **two different jobs**, so they are **two tools**:

```
city  ->  [ get_coordinates ]  ->  latitude, longitude  ->  [ get_weather ]  ->  the weather
```
The agent will **chain** them: it calls `get_coordinates`, **reads the numbers** in the answer, and then calls `get_weather` with those numbers. This is a **dependent** tool call: the second call needs the result of the first.

Both tools use **Open-Meteo**, which is free and needs no key.

### Tool 2: `get_coordinates`
The geocoding service returns a list in `"results"`. If the city does not exist there is **no** `"results"` key, so reading `["results"][0]` raises an error, and the `except` returns an `Error:` text. One `try / except`, no `if`.

In [ ]:
# TODO 2.1: write the tool.
#   - @tool, then  def get_coordinates(city: str) -> str:   and a one-sentence docstring (first line inside the function)
#   - put EVERYTHING inside ONE try / except Exception, and in the except return "Error: could not find the city ..."
#   Inside the try:
#     - place = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}, timeout=10).json()["results"][0]
#     - latitude = round(place["latitude"], 2)       longitude = round(place["longitude"], 2)       (2 decimals)
#     - return ONE line of text, for example:  Pokhara, Nepal: latitude 28.27, longitude 83.97
#       you need place["name"] and place.get("country", "").  Numbers must be turned into text with str(...) before you join them with +.
#   You need no "if": an unknown city has no "results", so ["results"][0] raises an error and the except catches it.

In [ ]:
# CHECK 2: run your tool by hand.
try:
    answer = get_coordinates.invoke({"city": "Pokhara"})
    print(answer)

    if answer.startswith("Error:"):
        print("NOT YET: the tool returned an error:", answer)
    elif "Pokhara" in answer and "latitude" in answer and "longitude" in answer:
        print("PASS: the tool found the city and its coordinates.")
    else:
        print("NOT YET: the line should have the city name, then latitude and longitude, for example: Pokhara, Nepal: latitude 28.27, longitude 83.97")

    unknown = get_coordinates.invoke({"city": "Qwxzzyville"})
    print(unknown)
    if unknown.startswith("Error:"):
        print("PASS: an unknown city returns an Error: text, not a crash.")
    else:
        print("NOT YET: an unknown city should return an Error: text. Check for a missing 'results' key.")
except NameError:
    print("NOT YET: write the tool in the cell above, and run that cell first.")

### Tool 3: `get_weather`
This tool takes the two **numbers** (not the city). It calls the Open-Meteo forecast service and returns one line. If the numbers make no sense (for example a latitude of 999), the service answers **without** a `"current"` key, so reading `weather["current"]` raises an error, and the `except` returns an `Error:` text. One `try / except`, no `if`.

In [ ]:
# TODO 2.2: write the tool.
#   - @tool, then  def get_weather(latitude: float, longitude: float) -> str:   and a one-sentence docstring
#     (say in the docstring that the numbers come from get_coordinates: the model reads it)
#   - put EVERYTHING inside ONE try / except Exception, and in the except return "Error: could not get the weather. ..."
#   Inside the try:
#     - weather = requests.get("https://api.open-meteo.com/v1/forecast", params={...}, timeout=10).json()
#       params: "latitude": latitude, "longitude": longitude, "current": "temperature_2m,wind_speed_10m",
#               "daily": "precipitation_probability_max", "forecast_days": 1
#     - temperature = weather["current"]["temperature_2m"]       wind = weather["current"]["wind_speed_10m"]
#     - rain = weather["daily"]["precipitation_probability_max"][0]
#     - return ONE line of text starting with the coordinates, for example:
#         At (28.27, 83.97): 17.5 C, wind 1.3 km/h, chance of rain today 25 percent.
#       Numbers must be turned into text with str(...) before you join them with +.
#   You need no "if": wrong numbers give an answer without "current", so weather["current"] raises an error and the except catches it.

In [ ]:
# CHECK 3: run your tool by hand, with the coordinates of Pokhara.
try:
    answer = get_weather.invoke({"latitude": 28.27, "longitude": 83.97})
    print(answer)

    if answer.startswith("Error:"):
        print("NOT YET: the tool returned an error:", answer)
    elif "28.27" in answer and " C" in answer and "rain" in answer:
        print("PASS: the tool returned the weather for the coordinates.")
    else:
        print("NOT YET: the line should start with the coordinates and have the temperature in C and the chance of rain.")

    wrong = get_weather.invoke({"latitude": 999, "longitude": 999})
    print(wrong)
    if wrong.startswith("Error:"):
        print("PASS: wrong coordinates return an Error: text, not a crash.")
    else:
        print("NOT YET: wrong coordinates should return an Error: text. Check for a missing 'current' key.")
except NameError:
    print("NOT YET: write the tool in the cell above, and run that cell first.")

---
# Part 3 - Tool 4: `convert_currency`

Models cannot know today's exchange rate, and they make arithmetic mistakes. So the tool gets the **live rate** and does the maths in Python. We use **open.er-api.com**: free, no key, and it includes Nepali rupees.

```
https://open.er-api.com/v6/latest/USD   ->   {"result": "success", "rates": {"NPR": 154.19, "EUR": 0.89, ...}}
```
The address ends with the currency you convert **from**. The `"rates"` dictionary holds the rate for every other currency. A wrong code gives an answer **without** `"rates"`, so reading `data["rates"][to_code]` raises an error, and the `except` returns an `Error:` text. One `try / except`, no `if`.

In [ ]:
# TODO 3.1: write the tool.
#   - @tool, then  def convert_currency(amount: float, from_code: str, to_code: str) -> str:   and a one-sentence docstring
#   - make both codes capital letters:  from_code = from_code.upper()   (do the same for to_code)
#   - put the rest inside ONE try / except Exception, and in the except return "Error: could not convert. ..."
#   Inside the try:
#     - data = requests.get("https://open.er-api.com/v6/latest/" + from_code, timeout=10).json()
#     - rate = data["rates"][to_code]
#     - result = round(amount * rate, 2)
#     - return ONE line of text, for example:  100.0 USD = 15419.47 NPR (1 USD = 154.19 NPR)
#       Use str(...) on every number before joining with +.
#   You need no "if": a wrong code gives no "rates" (or no such currency), so data["rates"][to_code] raises an error and the except catches it.

In [ ]:
# CHECK 4: run your tool by hand.
try:
    answer = convert_currency.invoke({"amount": 100, "from_code": "usd", "to_code": "npr"})
    print(answer)

    if answer.startswith("Error:"):
        print("NOT YET: the tool returned an error:", answer)
    elif "USD" in answer and "NPR" in answer:
        print("PASS: the tool converted the money. (Small letters worked too.)")
    else:
        print("NOT YET: the line should show both currency codes, for example 100 USD = ... NPR")

    wrong = convert_currency.invoke({"amount": 100, "from_code": "USD", "to_code": "XXX"})
    print(wrong)
    if wrong.startswith("Error:"):
        print("PASS: a wrong currency code returns an Error: text, not a crash.")
    else:
        print("NOT YET: a wrong code should return an Error: text. Check that to_code is in data['rates'].")
except NameError:
    print("NOT YET: write the tool in the cell above, and run that cell first.")

---
# Part 4 - The agent with five tools

Now one agent gets all five tools. The **system prompt** is its job description. Write it so the model knows **which tool to use for what**. This matters more as tools are added: with five tools, the model must choose.

A good prompt for this agent says:
- who it is (a trip-planning assistant) and today's date,
- which tool is for which job (find pages, read a page, coordinates, weather, money),
- that for the weather it must call `get_coordinates` **first**, then `get_weather` with the numbers from the answer, and never guess coordinates,
- to **search first, then read the best page** with `extract_page` before answering questions about places, using a **link from the search results**, never a made-up link,
- never to work out an exchange rate or the weather itself,
- to end with the link of the page it used,
- what to do when a tool returns an `Error:`,
- that its only tools are the four it has (the model sometimes asks for tools that do not exist).

In [ ]:
# TODO 4.1: write the system prompt as one text. Join the parts with + and " " (or write it in brackets over several lines).
#   Include, in your own words:
#   - you are a helpful trip-planning assistant, and today's date is (use the variable `today`)
#   - you have five tools; for the weather: first get_coordinates (city), then get_weather with the latitude and longitude it returns; never guess coordinates
#   - which tool is for money (convert_currency), which finds pages (search_web), which reads one page (extract_page)
#   - for places, things to do or plans: search first, then READ the best page with extract_page before answering
#   - extract_page needs a link taken from the search results: never invent a link
#   - never work out an exchange rate yourself: always use convert_currency
#   - never guess what a tool can tell you; do not add a year to search words unless the user gave one
#   - your only tools are search_web, extract_page, get_coordinates, get_weather and convert_currency
#   - end with the link of the page you used, written as plain text
#   - if extract_page returns an Error, try one other link from the search results
#   - if another tool returns an Error, say what failed and ask for different words
#   - answer in four to six short sentences
#   Start like this:   system_prompt = ("You are a helpful trip-planning assistant. Today's date is " + today + ". "

In [ ]:
# TODO 4.2: build the agent and ask it a question.
#   1. agent = create_agent(llm, tools=[search_web, extract_page, get_coordinates, get_weather, convert_currency], system_prompt=system_prompt)
#   2. question = "I have 50,000 NPR for a trip to Pokhara. What is the weather there today, and how much is that in US dollars?"
#   3. inside try / except Exception as error:   print("Stopped early:", str(error)[:200])
#        result = agent.invoke({"messages": [("user", question)]}, config={"recursion_limit": 12})
#        print(result["messages"][-1].content)       (the last message is the answer)

### Reading the answer
The agent printed only its **answer**. Behind the scenes it chose the tools from their **docstrings** and your **system prompt**, ran them, read the results, and wrote the answer from them. In Part 6 you will print **which tools** it used.

If it picks the wrong tool, improve one of those two things. If you see `Stopped early`, the model made a mistake (for example, it asked for a tool it does not have) or a service said "slow down". Run the cell again.

---
# Part 5 - Limits and memory

**Limits.** `recursion_limit` stops the agent after a number of steps. A trip question needs about 3 to 7 steps (model, tool, model, tool...), so 12 leaves room without letting it run forever.

**Memory.** On Day 2 you saw that a model forgets. An agent can keep its messages with a **saver** and a **thread id**: the name of the conversation. The same thread id brings the same memory back.

Now the agent gets a saver, and we ask a question and then a **follow-up** that only makes sense if the agent remembers.

In [ ]:
# TODO 5.1: give the agent a memory.
#   - create the agent again, like in 4.2, with ONE more argument:  checkpointer=InMemorySaver()
#   - config = {"configurable": {"thread_id": "trip-1"}, "recursion_limit": 12}
#     thread_id is the name of the conversation. Any text works.

In [ ]:
# This cell was written for you. Two questions in ONE thread: the same config is used for both,
# so the agent still has the first question and its answer when it gets the second.
questions = ["I have 50,000 NPR and 3 days in Pokhara. What is the weather, how much is that in USD, and what is one thing to do?",
             "Which activity did you suggest, and what is the weather there?"]

for question in questions:
    print("YOU:", question)
    try:
        result = agent.invoke({"messages": [("user", question)]}, config=config)
        print("AGENT:", result["messages"][-1].content)
    except Exception as error:
        print("Stopped early:", str(error)[:200])
    print()

The second question never says **where** or **which activity**. The agent knew, because it **still had the first question and its answer** in its messages. Now the same follow-up in a **new thread**:

In [ ]:
# A new thread name means a new, empty memory.
new_thread = {"configurable": {"thread_id": "trip-2"}, "recursion_limit": 12}

try:
    result = agent.invoke({"messages": [("user", "Which activity did you suggest, and what is the weather there?")]}, new_thread)
    print("New thread:", result["messages"][-1].content)       # without the history, the agent cannot know which place or activity
except Exception as error:
    print("Stopped early:", str(error)[:200])

---
# Part 6 - The class tests

Everybody runs the **same three questions**, so we can compare results. A new thread keeps the tests independent of each other. For each question, the cell prints the final answer and **which tools were used**.

| # | Question | Tools you expect |
|---|---|---|
| 1 | What is the weather in Pokhara today? | `get_coordinates`, then `get_weather` |
| 2 | Convert 50,000 NPR to USD, and then that amount in USD to EUR. | `convert_currency` (once or twice) |
| 3 | I have 50,000 NPR and 3 days in Pokhara. Give me the weather, the budget in USD and 3 things to do. | `get_coordinates`, `get_weather`, `convert_currency`, `search_web` and usually `extract_page` |

In [ ]:
tests = ["What is the weather in Pokhara today?",
         "Convert 50,000 NPR to USD, and then that amount in USD to EUR.",
         "I have 50,000 NPR and 3 days in Pokhara. Give me the weather, the budget in USD and 3 things to do."]

for number, question in enumerate(tests, 1):
    print("TEST", number, ":", question)
    tools_used = []
    try:
        thread = {"configurable": {"thread_id": "test-" + str(number)}, "recursion_limit": 12}     # a new thread for every test
        result = agent.invoke({"messages": [("user", question)]}, config=thread)
        print("   Answer:", result["messages"][-1].content)
        for message in result["messages"]:
            if message.type == "tool":                       # a message of type "tool" is a tool result
                tools_used.append(message.name)              # message.name is the name of the tool that ran
    except Exception as error:
        print("   Stopped early:", str(error)[:200])
    print("   Tools used:", tools_used)
    print()

### Your results table
Copy this table into a new markdown cell and fill it in.

| # | Worked? (yes / partly / no) | Tools used | What went wrong, if anything |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

**Find one failure and fix it.** Common fixes, from easiest:
1. **Wrong tool, or no tool:** make the tool's docstring clearer, or add a line to the system prompt.
2. **Wrong numbers:** check the tool by hand with the CHECK cell.
3. **`Stopped early`:** run the cell again, or raise `recursion_limit` a little.
4. **The page was not useful:** change the search words in the question.

Change **one** thing, run the test again, and write down what changed.

---
## What we learned
- Search and reading are **two tools**: `search_web` finds links, `extract_page` opens one. The agent decides to call the second one after the first.
- A weather service needs **coordinates**, so it is **two tools**: `get_coordinates` and `get_weather`. The agent **chains** them: the second call uses the numbers from the first.
- A money tool should do the **maths in Python**, with a live rate, and never in the model.
- Every tool returns an **`Error:` text** instead of crashing, so the agent can react.
- With five tools the **system prompt** matters: it says which tool is for which job, and which tool to call first.
- A **step limit** keeps the agent bounded. A **thread id** gives it memory for follow-up questions.

## Day 4
You will add more: a tool that saves the plan to a file, source links, and a checker that decides when the plan is complete.